# Laboratorio 10. Inicialización de pesos

**Pregunta guía:** ¿Por qué importa la escala inicial de los pesos en una red profunda?

## Objetivos
Al finalizar podrás comparar activaciones y gradientes bajo inicializaciones distintas; usar Xavier/Kaiming; mantener arquitectura y datos fijos.

## Prerrequisitos y conexión
Conecta la retropropagación (05) con las activaciones del laboratorio 09. Se requiere Python básico; cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Las inicializaciones buscan controlar varianzas a través de las capas. Para activaciones simétricas, Xavier usa aproximadamente $Var(W)=2/(fan_{in}+fan_{out})$; para ReLU, Kaiming usa $Var(W)=2/fan_{in}$. La activación y la conectividad determinan cuál es apropiada.

## Problema y datos
Una red profunda clasifica datos sintéticos no lineales bajo inicialización cero, Xavier y Kaiming. Todos los datos se generan en el notebook: no se requieren descargas ni archivos locales.

## Experimento conceptual
¿Qué evidencia muestran las activaciones, gradientes y pérdidas sobre la inicialización? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Inspeccionar propagación
Creamos la misma arquitectura profunda y registramos varianza de activaciones y norma de gradiente.


In [ ]:
X = torch.randn(256, 32, device=device)
y = (X[:, 0] * X[:, 1] + X[:, 2] > 0).long()
initialization_results = {}
for scheme in ['zeros', 'xavier', 'kaiming']:
    layers = nn.ModuleList([nn.Linear(32, 64), nn.Linear(64, 64), nn.Linear(64, 2)]).to(device)
    for layer in layers:
        if scheme == 'zeros': nn.init.zeros_(layer.weight)
        elif scheme == 'xavier': nn.init.xavier_uniform_(layer.weight)
        else: nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
        nn.init.zeros_(layer.bias)
    hidden1 = F.relu(layers[0](X)); hidden2 = F.relu(layers[1](hidden1)); logits = layers[2](hidden2)
    loss = F.cross_entropy(logits, y); loss.backward()
    initialization_results[scheme] = (hidden1.var().item(), hidden2.var().item(), layers[0].weight.grad.norm().item())
print('esquema: var capa1, var capa2, norma gradiente capa1')
for name, values in initialization_results.items(): print(name, np.round(values, 5))


**Resultado e interpretación.** La comparación de un forward/backward revela escalas y simetrías iniciales. No se ha entrenado todavía ni se concluye qué esquema generaliza mejor.


### Comparar curvas de aprendizaje
Entrenamos la misma arquitectura con inicializaciones distintas.


In [ ]:
training_histories = {}
for scheme in ['zeros', 'xavier', 'kaiming']:
    model = nn.Sequential(nn.Linear(32,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,2)).to(device)
    for layer in model:
        if isinstance(layer, nn.Linear):
            if scheme == 'zeros': nn.init.zeros_(layer.weight)
            elif scheme == 'xavier': nn.init.xavier_uniform_(layer.weight)
            else: nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
            nn.init.zeros_(layer.bias)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01); history = []
    for epoch in range(35):
        loss = F.cross_entropy(model(X), y); optimizer.zero_grad(); loss.backward(); optimizer.step(); history.append(loss.item())
    training_histories[scheme] = history
    print(scheme, 'exactitud train:', (model(X).argmax(1) == y).float().mean().item())
for name, history in training_histories.items(): plt.plot(history, label=name)
plt.xlabel('Época'); plt.ylabel('Cross-entropy'); plt.legend(); plt.show()


**Resultado e interpretación.** La curva es evidencia bajo una semilla, arquitectura y optimizador concretos. En la inicialización cero, neuronas equivalentes reciben gradientes simétricos y no aprenden roles distintos.


## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, tamaño de muestra, arquitectura y protocolo. No extrapoles métricas sintéticas a datos reales; separa decisiones de modelado de conclusiones generales. La celda de salida aporta la evidencia numérica y visual para responder la pregunta guía.

- Los parámetros y activaciones determinan cómo se representa la señal.
- La pérdida, el optimizador y el protocolo de evaluación forman parte del experimento.
- Una métrica aislada no describe todos los errores ni garantiza generalización.

## Ejercicios progresivos
1. **Guiado:** Guiado: sustituye ReLU por tanh y compara Xavier con Kaiming.
2. **Independiente:** Independiente: repite con tres semillas y resume variación de exactitud.
3. **Desafío:** Desafío: explica por qué una pérdida inicial similar no implica gradientes útiles en toda la profundidad.

## Conexión con el siguiente laboratorio
Usa la representación, operación o criterio de evaluación de este laboratorio como punto de partida del siguiente paso de la secuencia.
